# 1. EJEMPLO COMPARATIVO -> IRIS
El dataset Iris es un conjunto de datos que contiene información sobre flores del género Iris, concretamente de 3 especies:

- setosa
- versicolor
- virginica

Cada flor está descrita mediante 4 variables numéricas:

- Longitud del sépalo
- Anchura del sépalo
- Longitud del pétalo
- Anchura del pétalo

Y además tiene una etiqueta (la especie).
Puedes consultar más sobre este dataset en https://archive.ics.uci.edu/dataset/53/iris

En este notebook vamos a comparar dos enfoques distintos:

- un modelo de aprendizaje supervisado, que aprende usando etiquetas reales;
- un modelo K-Means, que agrupa observaciones sin conocer la especie.

La idea es comprobar que ambos pueden generar agrupaciones parecidas, pero no resuelven el mismo problema.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Cargar Iris

In [ ]:
# Mantén este código. En él se carga el Dataset como un Objeto
iris = load_iris()

X = iris.data
y = iris.target

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Nombres de las clases:", iris.target_names)

df = pd.DataFrame(X, columns=iris.feature_names)
df["clase"] = y
df["nombre_clase"] = df["clase"].map({
    0: iris.target_names[0],
    1: iris.target_names[1],
    2: iris.target_names[2]
})

df.head()

## 2.1 NOrmalización

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos normalizados (primeras filas):")
print(X_scaled[:5])

## 2.2 Método del Codo

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inercia = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X_scaled) # X_scaled es el dataset normalizado
    inercia.append(kmeans.inertia_)

plt.figure(figsize=(8,4))
plt.plot(range(1, 11), inercia, marker='o')
plt.title('Método del Codo')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.show()

# 3. Visualización sencilla de dos variables

Para representar el dataset de forma clara, vamos a usar dos variables:

- longitud del pétalo
- anchura del pétalo

Estas variables suelen separar mejor las especies que las medidas del sépalo.

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(X[:, 2], X[:, 3], c=y)
plt.xlabel("Longitud del pétalo")
plt.ylabel("Anchura del pétalo")
plt.title("Iris: clases reales")
plt.show()

# 4. Modelo Supervisado

## 4.1 Interpretación del modelo supervisado

En este caso, el modelo sí conoce las etiquetas reales durante el entrenamiento.

Por eso puede:

- aprender a distinguir especies;
- predecir la clase de nuevas flores;
- comparar sus predicciones con la realidad mediante métricas como la `accuracy` o la matriz de confusión.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

modelo_sup = LogisticRegression(max_iter=1000)
modelo_sup.fit(X_train, y_train)

y_pred = modelo_sup.predict(X_test)

print("Accuracy supervisado:", accuracy_score(y_test, y_pred))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nInforme de clasificación:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 4.2 Visualización prediciones supervisadas

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(X_test[:, 2], X_test[:, 3], c=y_pred)
plt.xlabel("Longitud del pétalo")
plt.ylabel("Anchura del pétalo")
plt.title("Predicciones del modelo supervisado")
plt.show()

# 5. K-Means

Ahora aplicamos **K-Means** sobre el mismo conjunto de datos, pero sin utilizar las etiquetas reales.

Esto significa que el algoritmo:

- no sabe qué flor es setosa, versicolor o virginica;
- solo agrupa observaciones según su similitud.

Por tanto, los clusters obtenidos no tienen por qué coincidir exactamente con las clases reales.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)
clusters = kmeans.labels_


print("Primeros clusters asignados:", clusters[:20])
print("\nCentroides:")
print(kmeans.cluster_centers_)

In [ ]:
# tengo que desnormalizar
centroides_original = scaler.inverse_transform(kmeans.cluster_centers_)

print("\nCentroides en escala original:")
print(centroides_original)

## 5.1 Visualización K - Means

In [ ]:
centroides_original = scaler.inverse_transform(kmeans.cluster_centers_)
# tenemos que poder medir en cm. K-Means "ve" datos normalizados; sin emabrgo, estamos representando centímetros
plt.figure(figsize=(7,5))
plt.scatter(X[:, 2], X[:, 3], c=clusters)
plt.scatter(
    centroides_original[:, 2],
    centroides_original[:, 3],
    marker='X',
    s=200,
    color='red'
)
plt.xlabel("Longitud del pétalo")
plt.ylabel("Anchura del pétalo")
plt.title("Clusters encontrados por K-Means")
plt.show()

# 6. Comparación

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.scatter(X[:, 2], X[:, 3], c=y)
plt.xlabel("Longitud pétalo")
plt.ylabel("Anchura pétalo")
plt.title("Clases reales")

plt.subplot(1,3,2)
plt.scatter(X_test[:, 2], X_test[:, 3], c=y_pred)
plt.xlabel("Longitud pétalo")
plt.ylabel("Anchura pétalo")
plt.title("Modelo supervisado")

plt.subplot(1,3,3)
plt.scatter(X[:, 2], X[:, 3], c=clusters)
plt.xlabel("Longitud pétalo")
plt.ylabel("Anchura pétalo")
plt.title("K-Means")

plt.tight_layout()
plt.show()

# 7. Comparación entre clases reales y clusters

La siguiente tabla permite comparar:

- las clases reales del dataset Iris;
- los grupos generados por K-Means.

Importante:
los números de los clusters no representan directamente las especies.
Por ejemplo, el cluster 0 no tiene por qué corresponder a la clase 0.

Esta tabla sirve para ver qué especies quedan mejor agrupadas y cuáles se mezclan más.

In [ ]:
nombres_clase = pd.Series(y).map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

tabla = pd.crosstab(
    pd.Series(nombres_clase, name="Clase real"),
    pd.Series(clusters, name="Cluster K-Means")
)

print(tabla)